# 6. PSFs from the Residual PSD with `PSFDataset`

Notebooks 1–5 make a PSF the direct way: draw an OPD screen with `PhaseDataset`, then image it with `WFS.GetPSF`. That gives one *instantaneous* (short-exposure) PSF per screen. A converged *long-exposure* PSF, the one a science camera integrates, needs the average of many screens.

`PSFDataset` skips the screens. Following AOPERA and maoppy, it builds the residual OPD **power spectral density** (PSD) of the AO loop analytically, from its error terms, and turns that PSD into the long-exposure PSF in one step:

$$\mathrm{OTF}(\rho) = \mathrm{OTF}_\mathrm{tel}(\rho)\; e^{-k^2\,(\sigma^2 - B(\rho))}\; \mathrm{OTF}_\mathrm{jitter}(\rho), \qquad B = \mathcal{F}[\Phi],\quad \sigma^2 = B(0),$$

where $\Phi$ is the residual OPD PSD, $B$ its covariance, and $k = 2\pi/\lambda_\mathrm{sci}$. One FFT turns the OTF into the PSF.

The module has two parts:

- **`PSFModel`**: the physics. It maps an explicit dict of parameters (r0, L0, wind, loop gain...) to a PSD, OTF, PSF and Strehl. It is fully differentiable, so it can also *fit* PSFs.
- **`PSFDataset`**: draws random parameters the same way `PhaseDataset` does and returns batches of PSFs.

The design and the math are in `Ideas/14-PSFDataset.md` at the repo root.

## Configuration

The instrument is the same synthetic one as in notebooks 1–3 (`wfs_params_exp.py`). `PSFDataset` reads the same four dicts `PhaseDataset` does, and draws r0, L0, layers, wind, loop gain/leak, level of correction and photons/RON from the same ranges.

PSF-only values go in one extra dict, **`PSFParams`**. The values below are **illustrative** for this fictional instrument, not a real camera:

- `Wavelength`: science wavelength. It defaults to the WFS wavelength.
- `sampling`: PSF pixels per λ/D, at least 2. It defaults to `WFSParams["sampling"]`.
- `fov`: field of view in λ/D, cropped exactly like `GetPSF` does.
- `NCPA`: rms range of a random static aberration map.
- `Jitter`: rms range of the residual tip-tilt jitter, per axis, in mas.
- `NoisePhotonFactor`, `NoiseRONFactor`: WFS noise sensitivity, `σ² = s_ph/n + s_ron (RON/n)²` rad² per WFS pixel. The values here are AOPERA's for a 4-face pyramid on a non-EMCCD camera.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mmengine import Config

from AI4AO import PhaseDataset, PSFDataset, PSFModel
from AI4AO.TorchPropagator import WFS

device = 'cuda' if torch.cuda.is_available() else 'cpu'

paramfile = 'wfs_params_exp.py'
config = Config.fromfile(paramfile)
WFSParams = config['WFSParams']
AtmosParams = config['AtmosParams']
LoopParams = config['LoopParams']
DMParams = config['DMParams']

# Illustrative PSF-side values for this synthetic instrument -- not a real camera.
PSFParams = {
    "Wavelength": 1.2e-6,     # science wavelength (m)
    "sampling": 3,            # PSF pixels per lambda/D (>= 2)
    "fov": 24,                # field of view (lambda/D)
    "NCPA": [10e-9, 40e-9],   # static aberration rms range (m)
    "Jitter": [1.0, 5.0],     # residual jitter rms per axis (mas)
    "NoisePhotonFactor": 1 / 4,
    "NoiseRONFactor": 1.0,
}

## Creating the dataset

`PSFDataset` takes a `WFS` instance. It uses only the WFS's pupil (with the central obstruction), `D`, `Nres` and the `GetPSF` convention. With every error term off, `PSFModel.PSF` reproduces `GetPSF` exactly. The plain `WFS` base class is enough, since no mask is needed.

`terms` selects the error terms, and all of them are on by default:

- `fitting`, `servo`: the same closed-loop PSD `PhaseDataset` uses (`generateClosedLoop = True`).
- `aliasing`: turbulence above the WFS sampling frequency folded into the corrected band.
- `noise`: WFS photon and read noise, propagated through the loop's noise transfer function.
- `static`: the NCPA map, applied inside the telescope OTF.
- `jitter`: a Gaussian tip-tilt OTF.

Every `psf_dataset[idx]` draws a fresh, independent batch. There is no time evolution, unlike `PhaseDataset`'s sequential indexing.

In [ ]:
wfs = WFS(WFSParams, device)
psf_dataset = PSFDataset(wfs, WFSParams, AtmosParams, LoopParams, DMParams, PSFParams, device,
                         return_short_exposure=True)
model = psf_dataset.model  # the PSFModel doing the physics

batch = psf_dataset[0]
for key, value in batch.items():
    print(f"{key:20s} {tuple(value.shape)}")

## Long-exposure PSFs

Each batch holds `Nphases` independent conditions. The Strehl ratio (`batch["strehl"]`) is the on-axis PSF value relative to the aberration-free pupil. The PSFs are shown on a log scale to bring out the halo. The corrected region is the dark square at `±(Nact − 1)/2` λ/D, and outside it the uncorrected fitting-error halo takes over.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for i, ax in enumerate(axes):
    ax.imshow(torch.log10(batch["psf"][i] / batch["psf"][i].max() + 1e-6).cpu(), vmin=-5, vmax=0)
    ax.set_title(f"r0 = {batch['r0'][i]:.3f} m, c = {batch['level_of_correction'][i]:.2f}\n"
                 f"Strehl = {batch['strehl'][i]:.2f}")
    ax.axis('off')
plt.suptitle("Long-exposure PSFs (log10, peak-normalized)")
plt.show()

## The error budget

`model.ResidualPSD(params, return_terms=True)` returns the total residual OPD PSD, plus each term separately, all in m²·m². The profiles below are cuts along `fx` for one sample, and the vertical line is the DM cutoff `fc = (Nact − 1)/(2D)`:

- Fitting error dominates above `fc`.
- Inside the band, servo-lag, aliasing and noise make up the residual.

Integrating each PSD gives its contribution to the wavefront error.

In [ ]:
params = psf_dataset.DrawRandomParameters()
total, terms = model.ResidualPSD(params, return_terms=True)

sample, c = 0, model.N // 2
f = model.fx[c:, c].cpu()
plt.figure(figsize=(7, 4.5))
for name, psd in terms.items():
    plt.loglog(f[1:], psd[sample, c + 1:, c].cpu(), label=name)
plt.loglog(f[1:], total[sample, c + 1:, c].cpu(), 'k--', label='total')
plt.axvline((DMParams['Nactuator'] - 1) / (2 * WFSParams['D']), color='gray', lw=0.8)
plt.xlabel('spatial frequency fx (1/m)')
plt.ylabel('OPD PSD (m$^2$ m$^2$)')
plt.legend()
plt.show()

print("Wavefront error budget, sample 0:")
for name, psd in {**terms, "total": total}.items():
    rms_nm = torch.sqrt(psd[sample].sum() * model.dF**2) * 1e9
    print(f"  {name:10s} {rms_nm:7.1f} nm rms")

## Long vs. short exposure

`return_short_exposure=True` adds `psf_short`: one *instantaneous* PSF per sample. It is built from a random residual screen drawn from the **same** PSD, plus the static map and a random jitter tilt, and imaged with `GetPSF`. Short exposures are independent draws, not a time series, and on average they converge to the long-exposure PSF. The right-hand panel shows the average of 50 of them.

In [ ]:
params = psf_dataset.DrawRandomParameters()
psd = model.ResidualPSD(params)
long_exposure = model.PSF(params)
short = [model.ShortExposurePSF(params, psd) for _ in range(50)]

panels = [(long_exposure[0], "long exposure"), (short[0][0], "one short exposure"),
          (torch.stack(short).mean(0)[0], "mean of 50 short exposures")]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (psf, title) in zip(axes, panels):
    ax.imshow(torch.log10(psf / long_exposure[0].max() + 1e-6).cpu(), vmin=-5, vmax=0)
    ax.set_title(title)
    ax.axis('off')
plt.show()

## Driving the model directly: Strehl vs. r0 and loop gain

`PSFModel` needs no random draws. Any dict of tensors with the documented keys works, and one sample with `B` entries gives `B` PSFs in parallel. `repeat_sample` below copies one drawn condition `n` times, so that a single parameter can then be swept across the batch.

In [ ]:
def repeat_sample(params, i, n):
    # n copies of sample i of a params dict. Layer quantities carry the batch
    # on their last axis, everything else on the first.
    out = {}
    for key, value in params.items():
        if key in ("fractional_r0", "layer_heights", "wind_speed", "wind"):
            out[key] = value[..., i:i + 1].expand(*value.shape[:-1], n).clone()
        else:
            out[key] = value[i:i + 1].expand(n, *value.shape[1:]).clone()
    return out

base = psf_dataset.DrawRandomParameters()
n = 25

sweep = repeat_sample(base, 0, n)
r0_values = torch.linspace(0.02, 0.15, n, device=device)
sweep["r0"] = r0_values
strehl_r0 = model.Strehl(model.OTF(sweep))

sweep = repeat_sample(base, 0, n)
gain_values = torch.linspace(0.05, 0.9, n, device=device)
sweep["loop_gain"] = gain_values
strehl_gain = model.Strehl(model.OTF(sweep))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(r0_values.cpu(), strehl_r0.cpu())
axes[0].set_xlabel('r0 at 500 nm (m)')
axes[1].plot(gain_values.cpu(), strehl_gain.cpu())
axes[1].set_xlabel('integrator gain')
for ax in axes:
    ax.set_ylabel(f"Strehl at {PSFParams['Wavelength'] * 1e6:.1f} $\\mu$m")
    ax.grid(alpha=0.3)
plt.show()

The gain curve shows the usual trade-off. Low gain cannot follow the turbulence, so servo-lag dominates. High gain amplifies noise, and past a point the loop goes unstable, where the noise transfer gain blows up.

## Cross-check against `PhaseDataset` + `GetPSF`

The long-exposure PSF should equal the average of `GetPSF` over many `PhaseDataset` closed-loop screens drawn under the same conditions. For an exact comparison, the check below uses:

- a second `PSFModel` restricted to the two terms `PhaseDataset` models (`fitting`, `servo`);
- at the WFS wavelength;
- with `sampling = 2`, the setting at which both sample exactly the same spatial frequencies.

`PSFModel.ParametersFromPhaseDataset` reads the conditions the `PhaseDataset` just drew, so each batch of screens is paired with its own analytic PSF.

Expect agreement to a few percent here. The residual is the speckle noise of a finite number of screens: these random conditions reach low Strehl, where speckle is strong. With pinned conditions and more screens, `tests/test_psf_dataset.py::test_matches_phase_dataset_ensemble` agrees to well under 1%.

In [ ]:
phase_dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
phase_dataset.generateClosedLoop = True
comparison_model = PSFModel(wfs, WFSParams, AtmosParams, LoopParams, DMParams, {"sampling": 2}, device,
                            terms=("fitting", "servo"))

n_draws = 60
screens_mean, analytic_mean = 0, 0
for _ in range(n_draws):
    opd = phase_dataset[0]["opd"]
    screens_mean = screens_mean + wfs.GetPSF(opd, sampling=2).mean(0) / n_draws
    params = PSFModel.ParametersFromPhaseDataset(phase_dataset)
    analytic_mean = analytic_mean + comparison_model.PSF(params).mean(0) / n_draws

c = comparison_model.N // 2
x = torch.arange(comparison_model.N) - c
plt.figure(figsize=(7, 4))
plt.semilogy(x / 2, screens_mean[c].cpu(), label=f'mean GetPSF over {n_draws * AtmosParams["Nphases"]} screens')
plt.semilogy(x / 2, analytic_mean[c].cpu(), '--', label='PSFModel (no screens)')
plt.xlabel('angle ($\\lambda$/D)')
plt.legend()
plt.show()
# On-axis pixel rather than the max: the max of a noisy (speckled) average is biased upward
print(f"on-axis ratio screens / analytic: {(screens_mean[c, c] / analytic_mean[c, c]).item():.3f}")
print(f"relative L1 difference: {((screens_mean - analytic_mean).abs().sum() / analytic_mean.sum()).item():.3f}")

## Fitting a PSF

Because `PSFModel` is differentiable end to end, it doubles as a parametric PSF model, in the spirit of maoppy's PSFAO. Below, a target PSF is generated with known conditions. r0 and the loop gain are then recovered by gradient descent from a wrong starting point, with every other parameter held at its true value. Each is reparameterized so the optimizer works on an unconstrained variable: `log r0`, and a logit for the gain in (0, 1). This is a few hundred optimizer steps on two scalars, not a network training run.

The target is pinned to a corrected regime: r0 = 8 cm, full correction, gain 0.3. In a seeing-limited PSF the fitting halo swamps the corrected band, so the PSF carries almost no information about the loop gain. Only r0 would then be recoverable.

In [ ]:
truth = repeat_sample(psf_dataset.DrawRandomParameters(), 0, 1)
truth["r0"] = torch.tensor([0.08], device=device)
truth["level_of_correction"] = torch.tensor([1.0], device=device)
truth["loop_gain"] = torch.tensor([0.3], device=device)
target = model.PSF(truth)

log_r0 = torch.tensor([np.log(0.2)], device=device, requires_grad=True)
gain_logit = torch.tensor([0.0], device=device, requires_grad=True)  # sigmoid(0) = 0.5
optimizer = torch.optim.Adam([log_r0, gain_logit], lr=0.05)

losses = []
for step in range(300):
    fit = dict(truth, r0=log_r0.exp(), loop_gain=torch.sigmoid(gain_logit))
    loss = ((model.PSF(fit) - target) ** 2).mean() / (target ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.semilogy(losses)
plt.xlabel('step')
plt.ylabel('relative MSE')
plt.show()
print(f"r0:   true {truth['r0'].item():.4f} m,  fitted {log_r0.exp().item():.4f} m")
print(f"gain: true {truth['loop_gain'].item():.3f},    fitted {torch.sigmoid(gain_logit).item():.3f}")

## Batch dict reference

`psf_dataset[idx]` returns (B = `AtmosParams["Nphases"]`, L = number of layers, drawn per batch):

- **`psf`** (B, H, W): the long-exposure PSF, in `GetPSF` units. H = W = `fov · sampling`, or the full grid when `fov` is `None`.
- **`psf_short`** (B, H, W): one short exposure, only with `return_short_exposure=True`.
- **`strehl`** (B,): on-axis Strehl ratio.
- **`psd`** (B, N, N): the total residual OPD PSD (m²·m²), centred.
- **The drawn conditions**, which are also the keys `PSFModel` accepts:
  - `r0`, `L0`, `level_of_correction`, `loop_gain`, `loop_leak`, `nphotons`, `ron` (B,);
  - `fractional_r0`, `layer_heights`, `wind_speed` (L, B); `wind` (2, L, B);
  - `ncpa_rms` (B,); `static_opd` (B, Nres, Nres); `jitter` (B, 2) in rad; `jitter_angle` (B,).

For a given seed, the atmosphere and loop draws are identical to `PhaseDataset`'s, since both use `PSD.DrawAOParameters`.